In [ ]:
# Space X Falcon 9 First Stage Landing Prediction
#
# Hands on Lab: Complete the Machine Learning Prediction lab
#
# Estimated time needed: 60 minutes
#
# Space X advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars;
# other providers cost upward of 165 million dollars each, much of the savings is because Space X
# can reuse the first stage. Therefore if we can determine if the first stage will land, we can
# determine the cost of a launch. This information can be used if an alternate company wants to
# bid against space X for a rocket launch. In this lab, you will create a machine learning pipeline
# to predict if the first stage will land given the data from the preceding labs.
#
# ## Objectives
#
# * Perform exploratory Data Analysis and determine Training Labels
# * Create a column for the class
# * Standardize the data
# * Split into training data and test data
# * Find best Hyperparameter for SVM, Classification Trees and Logistic Regression
# * Find the method that performs best using test data
#
# ## Import Libraries and Define Auxiliary Functions
#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

# Reproducibility
RANDOM_STATE = 2
np.random.seed(RANDOM_STATE)

# Compact plots
plt.rcParams["figure.figsize"] = (4, 4)
plt.rcParams["axes.grid"] = False

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Version-proof confusion matrix. Signature matches the lab text: (Y_test, yhat)."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels([0, 1])
    ax.set_yticklabels([0, 1])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    # Create colorbar
    plt.colorbar(im)
    plt.show()

print(f"Imports ready. RANDOM_STATE = {RANDOM_STATE}\n")

# Load the data
# We will load the features (X) and labels (Y) from the URLs provided in the notebook.
URL_DATASET_PART_2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
URL_DATASET_PART_3 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv"

#
# ### TASK 1
#
# Create a NumPy array from the column `Class` in `data`, by applying the method `to_numpy()`
# then assign it to the variable `Y`, make sure the output is a Pandas series
# (only one bracket `df['name of column']`).
#
print("--- TASK 1: Load Labels (Y) ---")
data = pd.read_csv(URL_DATASET_PART_2)
Y = data["Class"]

print(f"Y type: {type(Y)}")
print(f"Y shape: {Y.shape}")
print("Class distribution:\n", Y.value_counts(dropna=False))
print("\n")

#
# ### TASK 2
#
# Standardize the data in `X` then reassign it to the variable `X` using the `transform`
# provided below.
#
print("--- TASK 2: Load and Standardize Features (X) ---")
X = pd.read_csv(URL_DATASET_PART_3)
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)

print(f"X type after scaling: {type(X)}")
print(f"X standardized shape: {X.shape}")
print("\n")


# We split the data into training and testing data using the function `train_test_split`.
# The training data is divided into validation data, a second set used for training data;
# then the models are trained and hyperparameters are selected using the function `GridSearchCV`.
#
# ### TASK 3
#
# Use the function `train_test_split` to split the data `X` and `Y` into training and test data.
# Set the parameter `test_size` to `0.2` and `random_state` to `2`.
# The training data and test data should be assigned to the following labels.
# `X_train, X_test, Y_train, Y_test`
#
print("--- TASK 3: Split Data ---")
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=RANDOM_STATE
)
# Note: The original lab used stratify=Y in a helper cell, which is good practice.
# Let's stick to the prompt's exact request, but stratification is generally recommended.

print(f"Train shapes: {X_train.shape}, {Y_train.shape}")
print(f"Test shapes: {X_test.shape}, {Y_test.shape}")
print("\n")

# CV helper: choose a safe number of folds based on Y_train
# (This logic was in a helper cell in the notebook)
labels, counts = np.unique(np.asarray(Y_train).astype(np.int32), return_counts=True)
min_class_train = int(counts.min())
cv_splits = max(2, min(5, min_class_train))
print(f"Class counts in Y_train: {dict(zip(labels.tolist(), counts.tolist()))}")
print(f"Using cv = {cv_splits} for GridSearchCV")
print("\n")

#
# ### TASK 4
#
# Create a logistic regression object then create a `GridSearchCV` object `logreg_cv` with `cv = 10`.
# (Note: we are using `cv_splits` calculated above for safety, as 10 might be too high).
# Fit the object to find the best parameters from the dictionary `parameters`.
#
print("--- TASK 4: Logistic Regression ---")
parameters_lr = {"C": [0.01, 0.1, 1, 10], "penalty": ["l2"], "solver": ["lbfgs"]}
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg_cv = GridSearchCV(lr, parameters_lr, cv=cv_splits)
logreg_cv.fit(X_train, Y_train)

print(f"Tuned hyperparameters (best parameters): {logreg_cv.best_params_}")
print(f"Accuracy: {logreg_cv.best_score_}")
print("\n")

#
# ### TASK 5
#
# Calculate the accuracy on the test data using the method `score`:
#
print("--- TASK 5: Logistic Regression Evaluation ---")
test_acc_lr = logreg_cv.score(X_test, Y_test)
print(f"Test accuracy (LogReg): {test_acc_lr}")

# Lets look at the confusion matrix:
yhat_lr = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_lr, title="LogReg Confusion Matrix")
print("\n")


#
# ### TASK 6
#
# Create a support vector machine object then create a `GridSearchCV` object `svm_cv` with `cv = 10`.
# (Note: using `cv_splits` again).
# Fit the object to find the best parameters from the dictionary `parameters`.
#
print("--- TASK 6: Support Vector Machine (SVM) ---")
parameters_svm = {
    "kernel": ("linear", "rbf", "poly", "sigmoid"),
    "C": np.logspace(-2, 2, 5), # [0.01, 0.1, 1, 10, 100]
    "gamma": ["scale", "auto"]
}
svm = SVC(random_state=RANDOM_STATE)
svm_cv = GridSearchCV(svm, parameters_svm, cv=cv_splits)
svm_cv.fit(X_train, Y_train)

print(f"Tuned hyperparameters (best parameters): {svm_cv.best_params_}")
print(f"Accuracy: {svm_cv.best_score_}")
print("\n")

#
# ### TASK 7
#
# Calculate the accuracy on the test data using the method `score`:
#
print("--- TASK 7: SVM Evaluation ---")
test_acc_svm = svm_cv.score(X_test, Y_test)
print(f"Test accuracy (SVM): {test_acc_svm}")

# We can plot the confusion matrix
yhat_svm = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_svm, title="SVM Confusion Matrix")
print("\n")

#
# ### TASK 8
#
# Create a decision tree classifier object then create a `GridSearchCV` object `tree_cv` with `cv = 10`.
# (Note: using `cv_splits`).
# Fit the object to find the best parameters from the dictionary `parameters`.
#
print("--- TASK 8: Decision Tree ---")
parameters_tree = {
    "criterion": ["gini", "entropy"],
    "splitter": ["best", "random"],
    "max_depth": [2, 4, 6, 8, 10],
    "max_features": ["sqrt", None], # Corrected from "auto" which is deprecated to "sqrt"
    "min_samples_leaf": [1, 2, 4],
    "min_samples_split": [2, 5, 10]
}
tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_cv = GridSearchCV(tree, parameters_tree, cv=cv_splits)
tree_cv.fit(X_train, Y_train)

print(f"Tuned hyperparameters (best parameters): {tree_cv.best_params_}")
print(f"Accuracy: {tree_cv.best_score_}")
print("\n")

#
# ### TASK 9
#
# Calculate the accuracy of `tree_cv` on the test data using the method `score`:
#
print("--- TASK 9: Decision Tree Evaluation ---")
test_acc_tree = tree_cv.score(X_test, Y_test)
print(f"Test accuracy (Decision Tree): {test_acc_tree}")

# We can plot the confusion matrix
yhat_tree = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_tree, title="Decision Tree Confusion Matrix")
print("\n")

#
# ### TASK 10
#
# Create a k nearest neighbors object then create a `GridSearchCV` object `knn_cv` with `cv = 10`.
# (Note: using `cv_splits`).
# Fit the object to find the best parameters from the dictionary `parameters`.
#
print("--- TASK 10: K-Nearest Neighbors (KNN) ---")
parameters_knn = {
    "n_neighbors": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
    "p": [1, 2] # 1 = manhattan_distance, 2 = euclidean_distance
}
knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, parameters_knn, cv=cv_splits)
knn_cv.fit(X_train, Y_train)

print(f"Tuned hyperparameters (best parameters): {knn_cv.best_params_}")
print(f"Accuracy: {knn_cv.best_score_}")
print("\n")

#
# ### TASK 11
#
# Calculate the accuracy of `knn_cv` on the test data using the method `score`:
#
print("--- TASK 11: KNN Evaluation ---")
test_acc_knn = knn_cv.score(X_test, Y_test)
print(f"Test accuracy (KNN): {test_acc_knn}")

# We can plot the confusion matrix
yhat_knn = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_knn, title="KNN Confusion Matrix")
print("\n")

#
# ### TASK 12
#
# Find the method that performs best:
#
print("--- TASK 12: Model Comparison ---")
results = {
    "Logistic Regression": test_acc_lr,
    "SVM": test_acc_svm,
    "Decision Tree": test_acc_tree,
    "KNN": test_acc_knn,
}

print("Model test accuracies:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

best_model_name = max(results, key=results.get)
print(f"\nBest performing model: {best_model_name} with accuracy {results[best_model_name]:.4f}")

# Display as a DataFrame
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Test Accuracy"])
print("\n")
print(results_df.sort_values("Test Accuracy", ascending=False))

print("\nAll tasks replicated.")